In [3]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [4]:
print(os.getcwd())

d:\DE\RAG Project\incremental-rag-pipeline


In [5]:
from pathlib import Path
from pypdf import PdfReader
import re
import hashlib
import json

In [ ]:
DATA_DIR = Path("data")

batch_folders = sorted(
    [
        folder
        for folder in DATA_DIR.iterdir()
        if folder.is_dir() and folder.name.startswith("batch_")
    ],
    key=lambda folder: int(folder.name.split("_")[1])
)

pdf_files = []

for batch_folder in batch_folders:
    for pdf_path in batch_folder.glob("*.pdf"):
        pdf_files.append({
            "batch_id": batch_folder.name,
            "path": pdf_path
        })

print(f"Batches found: {len(batch_folders)}")
print(f"PDFs found: {len(pdf_files)}")

for item in pdf_files:
    print(item["batch_id"], "|", item["path"].name)

Batches found: 4
PDFs found: 13
batch_1 | azure_data_factory.pdf
batch_1 | data_engineering_fundamentals.pdf
batch_1 | mysql_for_data_engineering.pdf
batch_1 | rag_fundamentals.pdf
batch_2 | azure_databricks.pdf
batch_2 | data_modeling.pdf
batch_2 | pyspark_and_spark.pdf
batch_3 | advanced_rag.pdf
batch_3 | incremental_data_pipelines.pdf
batch_3 | python_for_data_engineering.pdf
batch_4 | apache_airflow.pdf
batch_4 | apache_kafka.pdf
batch_4 | data_lakehouse_architecture.pdf


In [7]:
documents = []

for item in pdf_files:
    pdf_path = item["path"]
    batch_id = item["batch_id"]

    reader = PdfReader(pdf_path)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    documents.append({
        "source_file": pdf_path.name,
        "batch_id": batch_id,
        "file_path": str(pdf_path),
        "text": text
    })

print(f"Documents extracted: {len(documents)}")

for doc in documents:
    print(
        doc["batch_id"],
        "|",
        doc["source_file"],
        "| characters:",
        len(doc["text"])
    )

Documents extracted: 13
batch_1 | azure_data_factory.pdf | characters: 4189
batch_1 | data_engineering_fundamentals.pdf | characters: 4169
batch_1 | mysql_for_data_engineering.pdf | characters: 3677
batch_1 | rag_fundamentals.pdf | characters: 4731
batch_2 | azure_databricks.pdf | characters: 3946
batch_2 | data_modeling.pdf | characters: 3572
batch_2 | pyspark_and_spark.pdf | characters: 3852
batch_3 | advanced_rag.pdf | characters: 4104
batch_3 | incremental_data_pipelines.pdf | characters: 3832
batch_3 | python_for_data_engineering.pdf | characters: 3797
batch_4 | apache_airflow.pdf | characters: 2867
batch_4 | apache_kafka.pdf | characters: 2742
batch_4 | data_lakehouse_architecture.pdf | characters: 2813


In [8]:
def clean_text(text):
    # Remove dataset header and page numbers
    text = re.sub(
        r"Incremental RAG Pipeline - Technical Learning Dataset",
        "",
        text
    )
    text = re.sub(r"Page \d+", "", text)

    # Remove metadata embedded inside the PDFs
    text = re.sub(r"Document ID\s*\nDOC\d+\s*", "", text)
    text = re.sub(r"Project\s*\n[^\n]+\s*", "", text)
    text = re.sub(r"Category\s*\n[^\n]+\s*", "", text)
    text = re.sub(r"Batch\s*\nbatch_\d+\s*", "", text)

   # Remove dataset-purpose paragraph
    text = re.sub(
        r"Purpose:.*?naturally produce multiple retrieval chunks\.",
        "",
        text,
        flags=re.DOTALL
    )

    # Clean extra whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Apply cleaning to every extracted document
for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Text cleaning completed.")

Text cleaning completed.


In [9]:
def generate_file_hash(file_path):
    with open(file_path, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()


for doc in documents:
    doc["content_hash"] = generate_file_hash(doc["file_path"])

print("Document hashes generated.")

for doc in documents:
    print(
        doc["batch_id"],
        "|",
        doc["source_file"],
        "|",
        doc["content_hash"][:12]
    )

Document hashes generated.
batch_1 | azure_data_factory.pdf | 8893e91aac5c
batch_1 | data_engineering_fundamentals.pdf | 3fae10f020eb
batch_1 | mysql_for_data_engineering.pdf | d4dd4782295a
batch_1 | rag_fundamentals.pdf | 576000a4988b
batch_2 | azure_databricks.pdf | b7abbb840e06
batch_2 | data_modeling.pdf | bcd5f34bcf74
batch_2 | pyspark_and_spark.pdf | c844b88ca7cf
batch_3 | advanced_rag.pdf | 085ce7e9744c
batch_3 | incremental_data_pipelines.pdf | eb2f03306dd3
batch_3 | python_for_data_engineering.pdf | e9d10d9c09c2
batch_4 | apache_airflow.pdf | 56dfc8e764d2
batch_4 | apache_kafka.pdf | 2da4a62497b3
batch_4 | data_lakehouse_architecture.pdf | 0213bf545117


In [10]:
DOCUMENT_REGISTRY_FILE = Path("audit/document_registry.json")

if DOCUMENT_REGISTRY_FILE.exists():
    with open(DOCUMENT_REGISTRY_FILE, "r", encoding="utf-8") as file:
        document_registry = json.load(file)
else:
    document_registry = {}

existing_numbers = [
    int(doc_id.replace("DOC", ""))
    for doc_id in document_registry.values()
]

next_doc_number = max(existing_numbers, default=0) + 1

for doc in documents:
    content_hash = doc["content_hash"]

    if content_hash in document_registry:
        doc["document_id"] = document_registry[content_hash]
    else:
        doc["document_id"] = f"DOC{next_doc_number}"
        document_registry[content_hash] = doc["document_id"]
        next_doc_number += 1

    doc["title"] = Path(doc["source_file"]).stem.replace("_", " ").title()

with open(DOCUMENT_REGISTRY_FILE, "w", encoding="utf-8") as file:
    json.dump(document_registry, file, indent=2)

print("Document metadata generated.")

for doc in documents:
    print(
        doc["document_id"],
        "|",
        doc["batch_id"],
        "|",
        doc["title"]
    )

Document metadata generated.
DOC1 | batch_1 | Azure Data Factory
DOC2 | batch_1 | Data Engineering Fundamentals
DOC3 | batch_1 | Mysql For Data Engineering
DOC4 | batch_1 | Rag Fundamentals
DOC5 | batch_2 | Azure Databricks
DOC6 | batch_2 | Data Modeling
DOC7 | batch_2 | Pyspark And Spark
DOC8 | batch_3 | Advanced Rag
DOC9 | batch_3 | Incremental Data Pipelines
DOC10 | batch_3 | Python For Data Engineering
DOC11 | batch_4 | Apache Airflow
DOC12 | batch_4 | Apache Kafka
DOC13 | batch_4 | Data Lakehouse Architecture


In [11]:
PROCESSED_FILE = Path("audit/processed_documents.json")

if PROCESSED_FILE.exists():
    with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
        processed_documents = json.load(file)
else:
    processed_documents = []

processed_hashes = {
    doc["content_hash"]
    for doc in processed_documents
}

next_batch_id = None

for batch_folder in batch_folders:
    batch_docs = [
        doc for doc in documents
        if doc["batch_id"] == batch_folder.name
    ]

    has_unprocessed_docs = any(
        doc["content_hash"] not in processed_hashes
        for doc in batch_docs
    )

    if has_unprocessed_docs:
        next_batch_id = batch_folder.name
        break

if next_batch_id:
    new_documents = [
        doc for doc in documents
        if doc["batch_id"] == next_batch_id
        and doc["content_hash"] not in processed_hashes
    ]

    print("Next batch:", next_batch_id)
    print("New documents:", len(new_documents))

    for doc in new_documents:
        print(
            doc["document_id"],
            "|",
            doc["title"]
        )
else:
    new_documents = []
    print("No new batches found.")

No new batches found.


In [12]:
OUTPUT_FILE = Path("audit/new_documents.json")

with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(new_documents, file, indent=2)

print(f"Saved {len(new_documents)} new documents for processing.")

Saved 0 new documents for processing.


In [14]:
from datetime import datetime

if next_batch_id is not None:

    audit_record = {
        "batch_id": next_batch_id,
        "documents_received": len(batch_docs),
        "new_documents": len(new_documents),
        "documents_skipped": len(batch_docs) - len(new_documents),
        "processing_timestamp": datetime.now().isoformat(timespec="seconds")
    }

    AUDIT_FILE = Path("audit/ingestion_audit.json")

    if AUDIT_FILE.exists():
        with open(AUDIT_FILE, "r", encoding="utf-8") as file:
            audit_history = json.load(file)
    else:
        audit_history = []

    audit_history.append(audit_record)

    with open(AUDIT_FILE, "w", encoding="utf-8") as file:
        json.dump(audit_history, file, indent=2)

    print("Audit record saved:")
    print(audit_record)

else:
    print("No unprocessed batch found. Audit record not created.")

No unprocessed batch found. Audit record not created.
